# QuizCadia question model: fine-tune with Unsloth

Teaches a small open model (Qwen3-4B, Apache 2.0) to write QuizCadia questions, then exports it as a
`q4_k_m` GGUF file (~2.5 GB) that runs on a normal CPU with llama.cpp.

**Before you start**
1. Run `python3 export_dataset.py` next to this notebook (it writes `data/train.jsonl` and `data/valid.jsonl`).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run the cells in order. Upload `train.jsonl`, `valid.jsonl` and `prompt.json` when asked.

Training on ~5,000 examples takes roughly 40-60 minutes on a free T4. At the end your browser downloads
`quizcadia-q4_k_m.gguf`; put it in `Quizcadia/ai/question-model/models/`.

Data licence: the imported questions come from Open Trivia DB (CC BY-SA 4.0). Keep that attribution with
the model you train.

## 1. Install Unsloth (Colab install from the official Unsloth notebooks)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 2. Upload the dataset and the prompt format

In [ ]:
import json, os
if not all(os.path.exists(f) for f in ("train.jsonl", "valid.jsonl", "prompt.json")):
    from google.colab import files
    files.upload()  # pick train.jsonl, valid.jsonl and prompt.json
PROMPT = json.load(open("prompt.json"))
print(sum(1 for _ in open("train.jsonl")), "training examples")

## 3. Load Qwen3-4B in 4-bit and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = 1024,   # questions are short; 1024 leaves plenty of room
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. Format the conversations

Qwen3 can "think" before answering. Quiz questions don't need that, so every user turn gets the
`/no_think` switch, exactly as the question generator sends it at run time.

In [ ]:
from datasets import load_dataset

def add_no_think(conversation):
    return [
        {**m, "content": m["content"] + PROMPT["no_think_suffix"]} if m["role"] == "user" else m
        for m in conversation
    ]

def to_text(batch):
    return {"text": [
        tokenizer.apply_chat_template(add_no_think(c), tokenize = False)
        for c in batch["conversations"]
    ]}

train = load_dataset("json", data_files = "train.jsonl", split = "train").map(to_text, batched = True)
valid = load_dataset("json", data_files = "valid.jsonl", split = "train")
train = train.shuffle(seed = 3407)
print(train[0]["text"])

## 5. Train (one pass over the data)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        padding_free = False,
    ),
)
trainer_stats = trainer.train()

## 6. Check it: generate questions for held-out topics

Counts how many replies are valid QuizCadia questions (JSON, right number of options, answer among them).
Facts still need a human check: in the app every generated question is a draft until an admin approves it.

In [ ]:
import json, random
FastLanguageModel.for_inference(model)

def ask(topic, difficulty, fmt):
    messages = add_no_think([
        {"role": "system", "content": PROMPT["system"]},
        {"role": "user", "content": PROMPT["user"].format(topic = topic, difficulty = difficulty, format = fmt)},
    ])
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True, enable_thinking = False)
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = 256, temperature = 0.7, top_p = 0.8, top_k = 20)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True).strip()

def is_valid(reply):
    try:
        q = json.loads(reply[reply.index("{"): reply.rindex("}") + 1])
        opts = q["options"]
        return (len(opts) in (2, 4) and len(set(o.lower() for o in opts)) == len(opts)
                and q["answer"] in opts and len(q["question"]) > 10)
    except Exception:
        return False

random.seed(1)
samples = [valid[i]["conversations"][1]["content"].split("\n") for i in random.sample(range(len(valid)), 20)]
ok = 0
for lines in samples:
    topic, difficulty, fmt = (l.split(": ", 1)[1] for l in lines[:3])
    reply = ask(topic, difficulty, fmt)
    ok += is_valid(reply)
    print(f"[{topic} / {difficulty}] {reply}\n")
print(f"{ok}/{len(samples)} replies are valid questions")

## 7. Export as GGUF (q4_k_m) and download

In [ ]:
import glob, os, shutil
model.save_pretrained_gguf("quizcadia_model", tokenizer, quantization_method = "q4_k_m")
# newest export wins, so a second run in the same runtime never picks up an older file
gguf = sorted((p for p in glob.glob("quizcadia_model*/*.gguf") if "Q4_K_M" in p.upper()), key = os.path.getmtime)
print(gguf)
shutil.copy(gguf[-1], "quizcadia-q4_k_m.gguf")

from google.colab import files
files.download("quizcadia-q4_k_m.gguf")

## 8. Run it with QuizCadia

Put the file at `Quizcadia/ai/question-model/models/quizcadia-q4_k_m.gguf`, then from `Quizcadia/`:

```bash
docker compose --profile ai up -d question-model question-generator
```

The admin page's **AI question drafts** section can then generate questions for any topic.